In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from jax import config
config.update("jax_enable_x64", True)
config.update("jax_platform_name", "cpu")

import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]=".8"

In [96]:
from tests.helpers import import_neuron_morph, jaxley2neuron_by_group, match_stim_loc
from jaxley.io.graph import swc_to_graph, from_graph, trace_branches, find_swc_trace_errors, simulate_swc_trace_errors, make_jaxley_compatible, add_missing_graph_attrs, split_branches, get_soma_idxs, branch_e2n, branch_n2e, insert_compartments, add_edge_lens
from jaxley.io.swc import swc_to_jaxley
from tests.helpers import import_neuron_morph, get_segment_xyzrL
import jaxley as jx
from jaxley.channels import HH
from jaxley.utils.cell_utils import v_interp

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

In [107]:
graph = swc_to_graph("tests/swc_files/morph.swc")
graph = add_missing_graph_attrs(graph)

# handles special case of a single soma node
if len(soma_idxs := get_soma_idxs(graph)) == 1:
    soma = soma_idxs[0]
    # Setting l = 2*r ensures A_cylinder = 2*pi*r*l = 4*pi*r^2 = A_sphere
    graph.add_node(-1, **graph.nodes[0])
    graph.add_edge(-1, soma, l=2 * graph.nodes[soma]["r"])
    graph = nx.relabel_nodes(graph, {i: i + 1 for i in graph.nodes})

    # edges connecting nodes to soma are considered part of the soma -> l = 0.
    for i, j in (*graph.in_edges(soma), *graph.out_edges(soma)):
        graph.edges[i, j]["l"] = 0

# ensure linear root segment to ensure root branch can be created.
if graph.out_degree(0) > 1:
    graph.add_node(-1, **graph.nodes[0])
    graph.add_edge(-1, 0, l=0.1)
    graph = nx.relabel_nodes(graph, {i: i + 1 for i in graph.nodes})

graph = trace_branches(graph)

graph = insert_compartments(graph, 4)

# create subgraph w.o. edges
comp_nodes = list(nx.get_node_attributes(graph, "comp_index"))
compartment_graph = nx.subgraph(graph, comp_nodes).copy()
# remove all edges
compartment_graph.remove_edges_from(list(compartment_graph.edges))

# add inter-branch compartment edges
df = pd.DataFrame([graph.nodes[n] for n in comp_nodes])
nodes_in_branch = np.stack(df.groupby("branch_index").apply(lambda x: x.index.values))
inter_branch_edges = np.array([nodes_in_branch[:, -1][np.array(find_branch_edges(graph))[:, 0]], nodes_in_branch[:, 0][np.array(find_branch_edges(graph))[:, -1]]]).T
compartment_graph.add_edges_from(inter_branch_edges)

# add intra-branch compartment edges
df.index = comp_nodes
intra_branch_edges = df.groupby("branch_index").apply(lambda x: branch_n2e(x.index.values))
intra_branch_edges = np.stack(intra_branch_edges.values).reshape(-1, 2)
compartment_graph.add_edges_from(intra_branch_edges)

# compartment_graph = add_edge_lens(compartment_graph)

# re-enumerate in dfs from 0
mapping = {old: new for new, old in enumerate(nx.dfs_preorder_nodes(compartment_graph, 1))}
compartment_graph = nx.relabel_nodes(compartment_graph, mapping)

/tmp/ipykernel_299807/2784630246.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nodes_in_branch = np.stack(df.groupby("branch_index").apply(lambda x: x.index.values))
/tmp/ipykernel_299807/2784630246.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intra_branch_edges = df.groupby("branch_index").apply(lambda x: branch_n2e(x.index.values))


In [36]:
def find_branch_edges(graph: nx.DiGraph):
    """Get all linearly connected paths in a graph aka. branches.

    The graph is traversed depth-first starting from the source node.

    Args:
        graph: A networkx graph.
        source_node: node at which to start graph traversal. If "leaf", the traversal
            starts at the first identified leaf node.

    Returns:
        A list of linear paths in the graph. Each path is represented as an array of
        edges."""
    edges = []

    for i, j in nx.dfs_edges(graph, 0):
        if is_branching(graph, i):
            pre_node = next(iter(graph.predecessors(i)))
            pre_branch = graph.edges[pre_node, i]["branch_index"]
            post_branch = graph.edges[i, j]["branch_index"]
            if pre_branch != post_branch:
                edges += [(pre_branch, post_branch)]

    return edges

array([[  0,   1],
       [  2,   3],
       [  3,   4],
       [  4,   5],
       [  5,   6],
       [  6,   7],
       [  7,   8],
       [  8,   9],
       [  9,  10],
       [ 10,  11],
       [ 11,  12],
       [ 12,  13],
       [ 13,  14],
       [ 14,  15],
       [ 15,  16],
       [ 16,  17],
       [ 16,  18],
       [ 15,  19],
       [ 14,  20],
       [ 20,  21],
       [ 20,  22],
       [ 13,  23],
       [ 12,  24],
       [ 24,  25],
       [ 24,  26],
       [ 26,  27],
       [ 26,  28],
       [ 11,  29],
       [ 10,  30],
       [  9,  31],
       [  8,  32],
       [  7,  33],
       [ 33,  34],
       [ 33,  35],
       [  6,  36],
       [ 36,  37],
       [ 36,  38],
       [ 38,  39],
       [ 38,  40],
       [ 40,  41],
       [ 40,  42],
       [  5,  43],
       [ 43,  44],
       [ 43,  45],
       [ 45,  46],
       [ 45,  47],
       [ 47,  48],
       [ 47,  49],
       [  4,  50],
       [  3,  51],
       [  2,  52],
       [ 52,  53],
       [ 52,

In [ ]:
comp = SimpleComp()
comp.insert(HH())
comp.record()
clamp1 = -50.0 * jnp.ones((1000,))
clamp2 = -60.0 * jnp.ones((1000,))
comp.clamp("v", clamp1)

def simulate(clamp):
    data_clamps = comp.data_clamp(
        "v", clamp, None
    )  # should override the first clamp
    return jx.integrate(comp, data_clamps=data_clamps)

jitted_simulate = jax.jit(simulate)

# Clamp2 should override clamp1 here
s = jitted_simulate(clamp2)
assert np.all(s[:, 1:] == -60.0)

comp2 = SimpleComp()
comp2.insert(HH())
branch1 = jx.Branch(comp, 4)
branch2 = jx.Branch(comp2, 4)
cell = jx.Cell([branch1, branch2], [-1, -1])

# Apply the clamp1 to the second branch via clamp
cell[1, 0].clamp("v", clamp1)

# cell.delete_recordings()
# cell.branch(0).comp(0).record()
# cell.branch(1).comp(0).record()

# def simulate(clamp):
#     data_clamps = cell.branch(0).comp(0).data_clamp("v", clamp, None)
#     return jx.integrate(cell, data_clamps=data_clamps)

# jitted_simulate = jax.jit(simulate)

# # Apply clamp2 to the first branch via data_clamp
# s = jitted_simulate(clamp2)

# assert np.all(s[0, 1:] == -60.0)
# assert np.all(s[1, 1:] == -50.0)

Added 1 recordings. See `.recordings` for details.
Added 1 external_states. See `.externals` for details.
Added 1 external_states. See `.externals` for details.


In [182]:
branch_edges

[array([[0, 1]]),
 array([[1, 2],
        [2, 3],
        [3, 4],
        [4, 5],
        [5, 6],
        [6, 7],
        [7, 8],
        [8, 9]]),
 array([[ 9, 10],
        [10, 11],
        [11, 12],
        [12, 13],
        [13, 14],
        [14, 15],
        [15, 16],
        [16, 17],
        [17, 18],
        [18, 19],
        [19, 20],
        [20, 21],
        [21, 22],
        [22, 23],
        [23, 24],
        [24, 25],
        [25, 26],
        [26, 27],
        [27, 28],
        [28, 29],
        [29, 30],
        [30, 31],
        [31, 32],
        [32, 33],
        [33, 34]]),
 array([[34, 35],
        [35, 36],
        [36, 37],
        [37, 38],
        [38, 39],
        [39, 40],
        [40, 41],
        [41, 42],
        [42, 43],
        [43, 44]]),
 array([[44, 45]]),
 array([[45, 46]]),
 array([[46, 47],
        [47, 48]]),
 array([[48, 49],
        [49, 50],
        [50, 51],
        [51, 52],
        [52, 53],
        [53, 54],
        [54, 55],
        [55, 5

In [181]:
jaxley_branches

,branch_index,node_index,id,x,y,z,r,l
0,0,0,1,0.00,0.00,0.00,8.119,0.000000
1,0,1,1,0.00,0.00,0.00,8.119,0.100000
2,1,1,1,0.00,0.00,0.00,8.119,0.000000
3,1,2,1,1.85,-4.03,0.00,7.230,4.434343
4,1,3,1,1.98,-6.00,0.00,6.280,6.408628
...,...,...,...,...,...,...,...,...
2781,155,2626,3,137.48,97.80,33.36,0.550,167.574652
2782,155,2627,3,139.56,102.04,33.36,0.550,172.297363
2783,155,2628,3,139.26,106.13,30.31,0.550,177.408194
2784,155,2629,3,138.86,112.19,44.47,0.550,192.815634


In [151]:
jaxley_branches

,branch_index,id,x,y,z,r
0,0,1,0.00,0.00,0.00,8.119
1,0,1,0.00,0.00,0.00,8.119
2,1,1,0.00,0.00,0.00,8.119
3,1,1,1.85,-4.03,0.00,7.230
4,1,1,1.98,-6.00,0.00,6.280
...,...,...,...,...,...,...
2781,155,3,137.48,97.80,33.36,0.550
2782,155,3,139.56,102.04,33.36,0.550
2783,155,3,139.26,106.13,30.31,0.550
2784,155,3,138.86,112.19,44.47,0.550


I think this is finally ready for a first round of reviews.

This has become quite the mammoth PR, but the functionality it enables is neat imo. For a rundown see the updated [08_morphologies.ipynb](https://github.com/jaxleyverse/jaxley/blob/ded457f3d55cf92e4218eefd842ce32fa8b6651b/tutorials/08_morphologies.ipynb)

All tests are passing now, which turned out to be an immense amount of work, but the imported morphologies are similar enough to NEURON both at the compartment level (x,y,z,r,l) and they also simulate correctly. I have essentially cloned the tests in `tests/test_swc.py` and added them to `tests/test_graph.py`. I have kept the tests and methods separated from everything else. The import and export functionality also works correctly.

Notable changes are:
- new `io` submodule
    - with `swc_to_graph`, `from_graph` and `to_graph` functions
    - moved `read_swc` there from `cell.py`
    - added `io.graph`
    - added `__eq__` to `Module` (I needed this to test whether re-imported modules are the same as the original.)
    - `test_graph.py` matches voltages between NEURON and Jaxley by matching compartment centers, this makes the test much more robust. Could also be added to `test_swc.py` if desired (see `tests/helpers.py`).

- Some open questions are:
    - `__eq__` is only implicitely being tested, and I dont know whether there could be a more efficient implementation
    - there is a lot of overlap between `test_graph` and `test_swc`. We could potentially move them to `tests/io/` and reduce overlap.


Lemme know your thoughts. Would also be happy to go through this in person.

In [332]:
from jaxley.utils.cell_utils import interpolate_xyz

def _update_nodes_with_xyz1(cell):
    """Add xyz coordinates to nodes."""
    loc = np.linspace(0.5 / cell.nseg, 1 - 0.5 / cell.nseg, cell.nseg)
    xyz = (
        [interpolate_xyz(loc, xyzr).T for xyzr in cell.xyzr]
        if len(loc) > 0
        else [cell.xyzr]
    )
    # idcs = cell.nodes["comp_index"]
    # cell.nodes.loc[idcs, ["x", "y", "z"]] = np.vstack(xyz)
    return np.vstack(xyz)


from jaxley.utils.cell_utils import v_interp
def _update_nodes_with_xyz2(cell):
    """Add xyz coordinates to nodes."""
    num_branches = len(cell.xyzr)
    x = np.linspace(
        0.5 / cell.nseg,
        (num_branches * 1 - 0.5 / cell.nseg),
        num_branches * cell.nseg,
    )
    x += np.arange(num_branches).repeat(
        cell.nseg
    )  # add offset to prevent branch loc overlap
    dls = [np.insert(np.cumsum(np.sqrt(np.sum(np.diff(xyzr[:,:3], axis=0) ** 2, axis=1))), 0, 0) for xyzr in cell.xyzr]
    xp = np.hstack([dl / dl.max() + 2 * i for i, dl in enumerate(dls)])
    xyz = v_interp(x, xp, np.vstack(cell.xyzr)[:, :3])
    # idcs = cell.nodes["comp_index"]
    # cell.nodes.loc[idcs, ["x", "y", "z"]] = xyz.T
    return xyz.T

graph = swc_to_graph("../tests/swc_files/morph.swc")
cell = from_graph(graph,nseg=8)

In [335]:
_update_nodes_with_xyz2(cell)

/tmp/ipykernel_2822953/3403501724.py:29: RuntimeWarning: invalid value encountered in divide
  xp = np.hstack([dl / dl.max() + 2 * i for i, dl in enumerate(dls)])


Array([[         nan,          nan,          nan],
       [         nan,          nan,          nan],
       [         nan,          nan,          nan],
       ...,
       [124.84836765,  67.02673345,  33.42886532],
       [135.52969153,  87.66895971,  33.35338212],
       [139.16860368, 107.51465421,  33.54542965]], dtype=float64)